# Qwen3-VL-4B-Instruct — MMMU Baseline Evaluation (v4, Colab L4)

## v4에서 수정된 사항 (실행 시간 / 결과 보존)

v3로 400문항을 실행한 결과 **총 13시간**이 예상되어 Colab 세션 한도를 넘겼고, 도중에 연결이 끊겼습니다.
실측 로그를 역산해 원인을 특정했습니다.

**관측 1** — batch 3, 7이 50문항을 각각 14.7초 / 13.3초에 완료 → **정상 응답 1문항 비용은 0.27초**.
**관측 2** — 총 347.5분에서 기본 비용을 빼고 역산하면 총 0.83~1.24M 토큰이 생성됨
→ `max_tokens=32768` 상한까지 간 문항이 **400건 중 25~38건(6~10%)**. 나머지 90%+는 수십 토큰에서 종료.
즉 소수의 러너웨이 생성이 전체 시간을 지배했고, `max_num_seqs=2`(v3의 보수적 설정) 때문에
러너웨이 1건이 배치 전체를 붙잡았습니다.

| # | v3 문제 | v4 수정 | 근거 |
|---|---|---|---|
| 1 | `max_tokens=32768` → 러너웨이 1건당 14~27분 | **`max_tokens=2048`** | 원 스펙에서 **"생성 예산(max_new_tokens)은 자유로 설정"** 항목. 고정 대상(temperature/top_p/top_k/repetition_penalty/presence_penalty/seed)은 공식값 그대로 유지 |
| 2 | `max_num_seqs=2` → 동시성 부족이 병목의 절반 | **`max_num_seqs=8`** + `enforce_eager=False` | `max_model_len` 49,152→18,432로 KV 4.2 GiB 확보 |
| 3 | 결과가 `/content`(휘발성)에만 저장 | **Google Drive에 저장** | 세션이 끊겨도 결과 보존 |
| 4 | `use_tqdm=False` → 배치당 최대 2시간 무출력 | **`use_tqdm=True`**, `BATCH_SIZE` 50→25 | 진행 가시성 + 체크포인트 빈도 |
| 5 | 설정이 다른 실행 결과가 섞일 위험 | **`run_config.json` 지문 검사** | 조건 불일치 시 실행 거부 |

**예상 소요 시간: 약 30분** (엔진 기동 3~8분 + 추론 16~18분 + 데이터셋 로드).

> **생성 예산 변경에 대한 명시**: 공식 recipe의 `out_seq_length=32768` 대신 2048을 사용합니다.
> 이는 원 지시사항이 자유 설정으로 지정한 항목이며, 상한 도달 건수는 `finish_reason='length'`로
> 로그에 남고 `summary.json`에 비율이 기록됩니다. 5%를 넘으면 경고가 출력됩니다.

## v3에서 수정된 사항 (torch/torchaudio CUDA 빌드 불일치)

**증상**: `RuntimeError: Detected that PyTorch and TorchAudio were compiled with different CUDA versions.
PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8`

**원인**: vLLM 0.29.0은 `torch==2.13.0`, `torchaudio==2.11.0`, `torchvision==0.28.0`, `torchcodec>=0.14`를
정확한 버전으로 고정합니다. PyPI의 `torch 2.13.0`은 **cu13 빌드**(`nvidia-*-cu13` 의존)인데,
Colab에 사전 설치된 `torchaudio`는 **버전 문자열이 똑같은 2.11.0이면서 cu128 빌드**입니다.
pip는 CUDA 빌드가 아니라 버전 문자열만 비교하므로 "이미 설치됨"으로 판단해 건너뛰었고,
그 결과 cu13 torch + cu128 torchaudio 조합이 만들어져 `import vllm` 시점에 터집니다.

**수정**:
1. 설치 전에 `torch torchvision torchaudio torchcodec`를 **먼저 제거**해, pip가 네 패키지를
   한 번의 해석으로 **동일 CUDA 빌드**로 새로 설치하도록 강제.
2. 재시작 직후 **torch 생태계 일관성 검증 셀** 추가 — 무거운 `import vllm` 전에 불일치를 먼저 잡습니다.
3. 불일치 시 vLLM 메타데이터에서 정확한 핀을 읽어 자동 복구하는 **복구 셀** 추가.

## v1 → v2에서 수정된 사항

| # | v1 문제 | 원인 | 수정 |
|---|---|---|---|
| 1 | `pip` 의존성 충돌 폭발, `pandas 3.0.6` 설치 | 설치 셀에서 `-U`로 `pillow`, `pandas`, `transformers`, `accelerate`, `qwen-vl-utils`를 불필요하게 강제 업그레이드 | vLLM과 `datasets`만 설치. `pandas` 의존 제거(표준 `csv` 사용), 미사용 패키지 제거 |
| 2 | `ImportError: cannot import name '_Ink' from 'PIL._typing'` | `-U pillow`가 **결함 릴리스 Pillow 12.3.0**을 설치함 | `pillow==12.2.0` 고정 + **설치 후 런타임 자동 재시작 셀** 추가 |
| 3 | (L4에서 확실히 실패할 설정) `max_model_len` 미지정 | 모델 기본 `max_position_embeddings=262144` → KV 캐시 **36.0 GiB** 요구 | `max_model_len=49152` 명시 + `enforce_eager=True` |
| 4 | 다중 이미지 문항에서 프롬프트 초과 가능 | 프로세서 기본 상한이 이미지 1장당 최대 16,384 비주얼 토큰 허용 | 이미지 면적 상한(`IMAGE_MAX_PIXELS`) — 원 스펙의 **"이미지 해상도는 자유 설정"** 항목 |
| 5 | 30개 config concat 스키마 위험 / 900문항 이미지 동시 적재 | 불필요한 concat | concat 제거, 배치 단위 지연 생성 |
| 6 | 배치 인덱스 기반 재개의 취약성 | `done // BATCH_SIZE` 방식 | **문항 id 기반 재개** |
| 7 | 전체 실행 전 검증 없음 | — | **스모크 테스트 셀** 추가 |

## 고정 스펙 (원 지시사항, 변경 없음)

| 항목 | 값 |
|---|---|
| 모델 | `Qwen/Qwen3-VL-4B-Instruct` |
| 모델 revision | `ebb281ec70b05090aa6165b016eac8ec08e71b17` |
| dtype | bf16 (리포지토리 기본값, 양자화/캐스팅 없음) |
| 데이터셋 | `MMMU/MMMU` |
| 데이터셋 revision | `98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68` |
| split | `validation`, 30개 config 전부 = 900문항 (필터링/서브샘플링 없음) |
| 하드웨어 | Colab Pro **L4 (24GB)** |
| 추론 엔진 | vLLM |

## Sampling recipe (Qwen 공식 문서, 임의 설정 아님)

출처: [QwenLM/Qwen3-VL README](https://github.com/QwenLM/Qwen3-VL) → **"Evaluation Reproduction"** 섹션
("To facilitate faithful reproduction of our reported results, we summarize our official evaluation settings below.")
[Qwen3-VL-4B-Instruct 모델 카드](https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct)의 "Generation Hyperparameters" 수치와 교차 확인.

```
greedy=false, seed=3407, top_p=0.8, top_k=20, temperature=0.7,
repetition_penalty=1.0, presence_penalty=1.5, out_seq_length(max_tokens)=32768
```

vLLM `SamplingParams`가 `presence_penalty`를 네이티브 지원하므로 recipe를 그대로 적용합니다.


## 1. 환경 설치

> **중요**
> 1. 이 셀은 **런타임당 한 번만** 실행합니다.
> 2. vLLM이 `torch`/`pillow`를 교체하므로, 설치 직후 **반드시 다음 셀(자동 재시작)을 실행**해야 합니다.
>    재시작 없이 `import vllm`을 하면 v1에서 겪은 `PIL._typing._Ink` 계열 오류가 발생합니다.
> 3. 재시작 후에는 **'torch 생태계 일관성 검증' 섹션부터** 순서대로 실행하세요 (설치 셀 재실행 불필요).
>
> 설치 중 출력되는 `cudf-cu12 / cuml-cu12 / dask-cuda ...` 의존성 경고는 Colab에 사전 설치된
> RAPIDS 스택과의 표면적 충돌이며, 본 평가(vLLM 추론)에는 영향이 없습니다.

In [ ]:
# [1/3] Colab 사전 설치된 torch 스택 제거
#   vLLM은 torch/torchaudio/torchvision/torchcodec를 정확한 버전으로 고정합니다.
#   Colab 사전 설치본은 "버전은 같지만 CUDA 빌드가 다른"(cu128) 휠이라
#   pip가 '이미 설치됨'으로 건너뛰어 cu13 torch + cu128 torchaudio 조합이 만들어집니다.
#   → 먼저 지워서 pip가 네 패키지를 한 번에 동일 빌드로 설치하도록 강제합니다.
!pip uninstall -y -q torch torchvision torchaudio torchcodec

# [2/3] vLLM 설치 (torch 스택을 일관된 CUDA 빌드로 함께 가져옵니다)
!pip install -q "vllm>=0.11.0" datasets

# [3/3] Pillow 12.3.0은 `from PIL._typing import _Ink`가 깨진 결함 릴리스이므로 직전 버전으로 고정
!pip install -q --force-reinstall --no-deps "pillow==12.2.0"

print("설치 완료 → 다음 셀(자동 재시작)을 실행하세요.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [ ]:
# 런타임 재시작 (설치된 torch/pillow를 깨끗하게 다시 로드하기 위해 필수)
print("런타임을 재시작합니다. 재시작이 끝나면 'torch 생태계 일관성 검증' 섹션부터 이어서 실행하세요.")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
except Exception:
    import os
    os.kill(os.getpid(), 9)


런타임을 재시작합니다. 재시작이 끝나면 'torch 생태계 일관성 검증' 섹션부터 이어서 실행하세요.


## 2. torch 생태계 일관성 검증 (재시작 직후 실행)

`import vllm`은 내부적으로 `torchaudio`/`torchvision`을 불러오고, 이들은 임포트 시점에
자신이 컴파일된 CUDA 버전과 torch의 CUDA 버전을 대조합니다. 불일치하면 13프레임짜리
`RuntimeError`가 나므로, **무거운 vLLM 임포트 전에 여기서 먼저** 확인합니다.

통과하면 그대로 다음 섹션으로, 실패하면 바로 아래 복구 셀을 실행하세요.

In [ ]:
import torch

print(f"torch        : {torch.__version__}  (CUDA {torch.version.cuda})")

problems = []
for name in ("torchvision", "torchaudio", "torchcodec"):
    try:
        mod = __import__(name)
        print(f"{name:13s}: {getattr(mod, '__version__', 'unknown')}  OK")
    except Exception as e:
        problems.append(name)
        print(f"{name:13s}: 실패 -> {type(e).__name__}: {str(e).splitlines()[0]}")

if problems:
    print("\n" + "=" * 80)
    print(f"torch 생태계 CUDA 빌드 불일치: {problems}")
    print("아래 '복구' 셀을 실행하세요 (자동으로 정확한 버전을 다시 설치하고 재시작합니다).")
    print("=" * 80)
else:
    print("\ntorch 생태계 일관성 OK → 다음 섹션으로 진행하세요.")


torch        : 2.13.0+cu130  (CUDA 13.0)
torchvision  : 0.28.0+cu130  OK
torchaudio   : 2.11.0+cu130  OK
torchcodec   : 0.16.0+cu130  OK

torch 생태계 일관성 OK → 다음 섹션으로 진행하세요.


### 복구 셀 (위 검증이 실패했을 때만 실행)

설치된 vLLM의 메타데이터에서 `torch` / `torchvision` / `torchaudio`의 **정확한 고정 버전**을 읽어
PyPI에서 동일 CUDA 빌드로 강제 재설치한 뒤 런타임을 재시작합니다. 재시작 후 위 검증 셀부터 다시 실행하세요.

In [ ]:
# 위 검증이 통과했다면 이 셀은 건너뛰세요.
from importlib.metadata import requires

pins = []
for r in (requires("vllm") or []):
    name = r.split("==")[0].split(";")[0].split("[")[0].strip()
    if name in ("torch", "torchvision", "torchaudio") and "==" in r:
        pins.append(r.split(";")[0].strip())

print("vLLM이 요구하는 정확한 버전:", pins)
assert pins, "vLLM 메타데이터에서 torch 핀을 찾지 못했습니다. 설치가 정상인지 확인하세요."

pin_args = " ".join(f'"{p}"' for p in pins)
get_ipython().system(f'pip install -q --force-reinstall --no-deps {pin_args} "torchcodec>=0.14"')

print("재설치 완료 → 런타임을 재시작합니다. 재시작 후 위 검증 셀부터 다시 실행하세요.")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
except Exception:
    import os
    os.kill(os.getpid(), 9)


## 3. 하드웨어 확인 (L4 24GB)

원 스펙의 하한은 RTX 4090(24GB)입니다. Colab Pro에서 **L4** 또는 **A100** 런타임을 선택하세요 (T4 16GB는 하한 미달).

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 유형을 L4 또는 A100으로 변경하세요."

gpu_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {gpu_name} | VRAM: {total_mem_gb:.1f} GiB | compute capability: {cc[0]}.{cc[1]}")

assert total_mem_gb >= 21.0, (
    f"최소 24GB급 GPU가 필요합니다 (원 스펙 하한: RTX 4090). "
    f"현재 {gpu_name} ({total_mem_gb:.1f} GiB)는 하한 미달입니다. L4 또는 A100을 선택하세요."
)
# bf16은 Ampere(8.0) 이상에서 네이티브 지원. L4 = 8.9, A100 = 8.0
assert cc[0] >= 8, f"bf16 네이티브 지원(compute capability >= 8.0)이 필요합니다. 현재 {cc[0]}.{cc[1]}"
print("하드웨어 요건 충족 (bf16 네이티브 지원)")


Sun Sep 20 17:20:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 4. 결과 저장 위치 (Google Drive)

v3에서는 결과를 `/content`에 저장해, 세션이 끊기면 5.8시간치 결과가 사라질 위험이 있었습니다.
v4는 Google Drive에 저장하므로 런타임이 회수돼도 결과가 남고, 재개도 그대로 동작합니다.

Drive를 쓰지 않으려면 `USE_DRIVE = False`로 바꾸세요 (휘발성 저장이 됩니다).

In [ ]:
USE_DRIVE = True   # False면 /content(휘발성)에 저장

BASE_DIR = "/content"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = "/content/drive/MyDrive"
        print("Google Drive 마운트 완료 → 세션이 끊겨도 결과가 보존됩니다.")
    except Exception as e:
        print(f"[경고] Drive 마운트 실패 ({e}) → /content에 저장합니다. 세션 종료 시 결과가 사라집니다.")
else:
    print("[경고] USE_DRIVE=False → /content(휘발성)에 저장합니다.")

print("BASE_DIR:", BASE_DIR)


Mounted at /content/drive
Google Drive 마운트 완료 → 세션이 끊겨도 결과가 보존됩니다.
BASE_DIR: /content/drive/MyDrive


## 5. 고정 스펙 상수 및 L4 메모리 산정

In [ ]:
import os

MODEL_REPO = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"

DATASET_REPO = "MMMU/MMMU"
DATASET_REVISION = "98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68"
DATASET_SPLIT = "validation"

MMMU_SUBJECTS = [
    "Accounting", "Agriculture", "Architecture_and_Engineering", "Art", "Art_Theory",
    "Basic_Medical_Science", "Biology", "Chemistry", "Clinical_Medicine", "Computer_Science",
    "Design", "Diagnostics_and_Laboratory_Medicine", "Economics", "Electronics",
    "Energy_and_Power", "Finance", "Geography", "History", "Literature", "Manage",
    "Marketing", "Materials", "Math", "Mechanical_Engineering", "Music", "Pharmacy",
    "Physics", "Psychology", "Public_Health", "Sociology",
]
assert len(MMMU_SUBJECTS) == 30

# ---------------------------------------------------------------------------
# Qwen3-VL 공식 "Evaluation Reproduction" recipe (Instruct 계열) — 임의 설정 금지 항목
# 출처: https://github.com/QwenLM/Qwen3-VL  README.md, "Evaluation Reproduction"
# ---------------------------------------------------------------------------
SAMPLING_RECIPE = dict(
    temperature=0.7,          # 공식값 고정
    top_p=0.8,                # 공식값 고정
    top_k=20,                 # 공식값 고정
    repetition_penalty=1.0,   # 공식값 고정
    presence_penalty=1.5,     # 공식값 고정
    seed=3407,                # 공식값 고정
    # [v5 변경] max_tokens 2048 → 8192
    #   v4(2048) 실측 결과:
    #     - finish_reason='length' 129건(14.3%)
    #     - 절단 문항 정확도 24.0% = 4지선다 무작위(25%)와 동일
    #       → MMMU 공식 파서가 보기 문자를 못 찾으면 random.choice로 폴백하기 때문
    #     - 정상 종료 771건은 62.0%, 그중 객관식만 65.6% (공식 보고치 67.4%와 -1.8%p)
    #   즉 격차의 대부분이 생성 예산 부족에서 왔으므로 상한을 4배로 올립니다.
    #   공식값 32768은 v3 실측에서 총 13시간이 걸려 Colab 세션 한도를 초과했습니다.
    max_tokens=8192,
)
OFFICIAL_MAX_TOKENS = 32768   # 기록용 (공식 recipe 원값)

# ---------------------------------------------------------------------------
# L4(24GB) 메모리 산정  — config.json @ 고정 revision 기준
#   text_config: num_hidden_layers=36, num_key_value_heads=8, head_dim=128, dtype=bfloat16
#   KV 캐시/토큰 = 2(K,V) x 36 x 8 x 128 x 2 bytes = 147,456 B = 144 KiB
#
#   * 모델 기본값 max_position_embeddings = 262,144 를 그대로 쓰면 36.0 GiB 필요 → L4 초기화 실패.
#     따라서 max_model_len 명시는 필수입니다.
#
#   [v5 변경] MAX_MODEL_LEN 18,432 → 12,288
#     v4 실행에서 900문항 프롬프트 토큰을 전부 측정한 결과:
#       중앙값 338 / 90분위 1,081 / 99분위 1,254 / 최대 2,491
#     v4의 프롬프트 예산 16,384는 실측 최대의 6.6배로 과다 배정이었습니다.
#     프롬프트 예산을 4,096(실측 최대의 1.6배)으로 줄이고 그만큼을 생성 예산에 돌립니다.
#
#   MAX_MODEL_LEN = 12,288 = 8,192(생성) + 4,096(프롬프트)
#     시퀀스 1개 최대 KV = 12,288 x 144 KiB = 1.69 GiB  (v4의 2.53 GiB보다 감소)
#     가중치(bf16, 약 4.4B) 약 8.8 GiB / L4 가용 22.5 GiB x 0.90 = 약 20.3 GiB
#     KV 가용 약 9.4 GiB = 약 68,800 토큰 → 전장 기준 동시 5.6개
#     실측 평균 시퀀스(프롬프트 474 + 생성 401 ≈ 875토큰) 기준으로는 여유가 매우 큼
# ---------------------------------------------------------------------------
MAX_MODEL_LEN = 12288

# ---------------------------------------------------------------------------
# 이미지 면적 상한 — v4와 동일 (변경 없음)
#   preprocessor_config.json: patch_size=16, merge_size=2 → 비주얼 토큰 1개 = 1,024 px
#   v4 실측: 984장 중 96장(9.8%)만 상한에 걸려 축소됨
# ---------------------------------------------------------------------------
IMAGE_MAX_PIXELS = 1_048_576

MAX_IMAGES_PER_QUESTION = 7
BATCH_SIZE = 25
IMAGE_CONTENT_MODE = "image_pil"

# [v5 변경] RUN_TAG 필수 변경 — 안 바꾸면 아래 지문 검사가 실행을 거부합니다.
#           v4 결과는 v4_maxtok2048 디렉터리에 그대로 보존됩니다.
RUN_TAG = "v5_maxtok8192"
OUTPUT_DIR = os.path.join(BASE_DIR, "mmmu_qwen3vl4b_eval", RUN_TAG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

RAW_LOG_PATH = os.path.join(OUTPUT_DIR, "raw_generations.jsonl")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "run_config.json")

# --- 실행 조건 지문: 설정이 다른 결과가 한 디렉터리에 섞이는 것을 막습니다 ---
import json

RUN_CONFIG = {
    "model_repo": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "split": DATASET_SPLIT,
    "sampling_recipe": SAMPLING_RECIPE,
    "max_model_len": MAX_MODEL_LEN,
    "image_max_pixels": IMAGE_MAX_PIXELS,
}

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, encoding="utf-8") as f:
        prev = json.load(f)
    if prev != RUN_CONFIG:
        _msg = [
            "[중단] 이 디렉터리에 다른 설정으로 만든 결과가 있습니다.",
            f"  경로: {OUTPUT_DIR}",
            f"  기존: {prev}",
            f"  현재: {RUN_CONFIG}",
            "→ RUN_TAG를 바꾸거나 해당 디렉터리를 비우고 다시 실행하세요.",
        ]
        raise SystemExit(chr(10).join(_msg))
    print("기존 실행과 설정 일치 → 이어서 실행 가능")
else:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(RUN_CONFIG, f, ensure_ascii=False, indent=2)
    print("새 실행 디렉터리 생성")

print("OUTPUT_DIR:", OUTPUT_DIR)
print(f"생성 예산 {SAMPLING_RECIPE['max_tokens']:,} / 프롬프트 예산 "
      f"{MAX_MODEL_LEN - SAMPLING_RECIPE['max_tokens']:,} (v4 실측 최대 프롬프트 2,491)")

기존 실행과 설정 일치 → 이어서 실행 가능
OUTPUT_DIR: /content/drive/MyDrive/mmmu_qwen3vl4b_eval/v5_maxtok8192
생성 예산 8,192 / 프롬프트 예산 4,096 (v4 실측 최대 프롬프트 2,491)


## 6. 모델 로드 (vLLM)

- `revision`을 모델·토크나이저 양쪽에 고정합니다.
- `dtype="bfloat16"` — 리포지토리 기본값 그대로, 양자화 없음.
- `max_model_len=18432` — 생성 예산 2,048 + 프롬프트 예산 16,384. 위 셀의 산정 근거 참조.
- `max_num_seqs=8`, `enforce_eager=False` — v3의 `max_num_seqs=2`가 병목의 절반이었습니다.
  `max_model_len` 축소로 확보한 KV 여유를 동시성에 투입합니다.
  초기화 중 OOM이 나면 `enforce_eager=True` → 그래도 부족하면 `max_num_seqs=4`로 낮추세요.
- min_pixels/max_pixels는 `mm_processor_kwargs`로 건드리지 않습니다 → **프로세서 기본값 유지**.
  (해상도 조정은 '이미지 처리' 섹션에서 데이터 측 면적 상한으로만 수행)

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=MODEL_REPO,
    revision=MODEL_REVISION,
    tokenizer_revision=MODEL_REVISION,
    dtype="bfloat16",
    trust_remote_code=True,
    limit_mm_per_prompt={"image": MAX_IMAGES_PER_QUESTION},
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=0.90,
    max_num_seqs=8,          # v3의 2 → 8. MAX_MODEL_LEN 축소로 확보한 KV 여유를 동시성에 사용.
    enforce_eager=False,     # CUDA 그래프 사용(약 10~15% 가속).
                             #   만약 초기화 중 OOM이 나면 enforce_eager=True로 되돌리고,
                             #   그래도 부족하면 max_num_seqs를 4로 낮추세요.
)

sampling_params = SamplingParams(
    temperature=SAMPLING_RECIPE["temperature"],
    top_p=SAMPLING_RECIPE["top_p"],
    top_k=SAMPLING_RECIPE["top_k"],
    repetition_penalty=SAMPLING_RECIPE["repetition_penalty"],
    presence_penalty=SAMPLING_RECIPE["presence_penalty"],
    seed=SAMPLING_RECIPE["seed"],
    max_tokens=SAMPLING_RECIPE["max_tokens"],
)
print(sampling_params)


INFO 09-20 17:21:07 [api_utils.py:286] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 12288, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 8, 'disable_log_stats': True, 'revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17', 'tokenizer_revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17', 'limit_mm_per_prompt': {'image': 7}, 'model': 'Qwen/Qwen3-VL-4B-Instruct'}


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

INFO 09-20 17:21:22 [model.py:684] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 09-20 17:21:22 [model.py:2021] Using max model len 12288


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

WARNING 09-20 17:21:22 [model.py:981] Model does not support mm_device_do_normalize, forcing mm_device_do_normalize = False.


...

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 09-20 17:21:42 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 09-20 17:25:47 [hf.py:547] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


[transformers] Qwen3VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


INFO 09-20 17:26:06 [base.py:244] Multi-modal warmup completed in 19.484s
INFO 09-20 17:26:08 [base.py:244] Readonly multi-modal warmup completed in 2.147s
SamplingParams(n=1, presence_penalty=1.5, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.7, top_p=0.8, top_k=20, min_p=0.0, seed=3407, stop=[], stop_token_ids=[], bad_words=[], thinking_token_budget=None, include_stop_str_in_output=False, ignore_eos=False, max_tokens=8192, min_tokens=0, logprobs=None, prompt_logprobs=None, skip_special_tokens=True, spaces_between_special_tokens=True, structured_outputs=None, extra_args=None)


## 7. 데이터셋 로드 (30개 config 전부, 서브샘플링 없음)

v1과 달리 `concatenate_datasets`를 사용하지 않습니다. 과목별 Dataset을 그대로 유지하고
`(과목, 행 인덱스, 문항 id)` 평면 인덱스만 만들어 두었다가, 실제 샘플(이미지 포함)은 배치 단위로 지연 생성합니다.
→ config 간 스키마 불일치 위험 제거 + 900문항 이미지 동시 적재로 인한 RAM 낭비 제거.

In [ ]:
from datasets import load_dataset

subject_datasets = {}
flat_index = []   # (subject, row_idx, example_id)

for subject in MMMU_SUBJECTS:
    ds = load_dataset(
        DATASET_REPO,
        subject,
        split=DATASET_SPLIT,
        revision=DATASET_REVISION,
    )
    subject_datasets[subject] = ds
    ids = ds["id"]                      # id 컬럼만 읽으므로 이미지 디코딩 없음
    for row_idx, ex_id in enumerate(ids):
        flat_index.append((subject, row_idx, ex_id))
    print(f"{subject:40s} n={len(ds)}")

print("\ntotal:", len(flat_index))
assert len(flat_index) == 900, (
    f"기대한 900문항이 아니라 {len(flat_index)}문항이 로드되었습니다. "
    f"필터링/서브샘플링 없이 원본 그대로인지 확인하세요."
)
assert len(set(i[2] for i in flat_index)) == 900, "문항 id에 중복이 있습니다."


README.md:   0%|          | 0.00/42.7k [00:00<?, ?B/s]

Accounting/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  273kB            

Accounting/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Accounting/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.54MB            

Accounting/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

Sociology/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/252 [00:00<?, ? examples/s]

Sociology                                n=30

total: 900


## 8. MMMU 공식 프롬프트 템플릿 / 파싱 / 채점 로직

아래 함수들은 MMMU 벤치마크 공식 레퍼런스 구현을 그대로 이식한 것입니다 (채점 방식에 의한 편차 제거).

- 원본: [MMMU-Benchmark/MMMU](https://github.com/MMMU-Benchmark/MMMU)
  - `mmmu/utils/eval_utils.py` → `parse_multi_choice_response`, `parse_open_response`,
    `eval_multi_choice`, `eval_open`, `normalize_str`, `extract_numbers`
  - `mmmu/utils/data_utils.py` → `construct_prompt`
  - `mmmu/configs/llava1.5.yaml` → 프롬프트 템플릿 문자열

`parse_gold_answer`만 이 노트북에서 추가한 방어적 유틸입니다 (open 문항 정답이 `"['42', '42.0']"` 형태의
복수 허용 문자열로 저장된 경우 복원 / 파싱 실패 시 원본 문자열 그대로 사용).

In [ ]:
import ast
import random
import re

random.seed(42)
import numpy as np

TASK_INSTRUCTIONS = ""
MULTI_CHOICE_EXAMPLE_FORMAT = "{}\n\n{}\n\nAnswer with the option's letter from the given choices directly."
SHORT_ANS_EXAMPLE_FORMAT = "{}\n\nAnswer the question using a single word or phrase."


def parse_gold_answer(sample):
    """open 문항의 gold answer가 리스트 형태 문자열인 경우 복원 (이 노트북 추가분)."""
    ans = sample["answer"]
    if sample["question_type"] == "multiple-choice":
        return ans
    try:
        parsed = ast.literal_eval(ans)
        return parsed if isinstance(parsed, list) else ans
    except (ValueError, SyntaxError):
        return ans


def construct_prompt(sample):
    """MMMU 공식 data_utils.py::construct_prompt 이식."""
    question = sample["question"]
    options = ast.literal_eval(sample["options"]) if isinstance(sample["options"], str) else sample["options"]
    res = {}
    if sample["question_type"] == "multiple-choice":
        start_chr = "A"
        example = ""
        prediction_range = []
        index2ans = {}
        for option in options:
            prediction_range.append(start_chr)
            example += f"({start_chr}) {option}\n"
            index2ans[start_chr] = option
            start_chr = chr(ord(start_chr) + 1)
        empty_prompt = MULTI_CHOICE_EXAMPLE_FORMAT.format(question, example)
        res["index2ans"] = index2ans
        res["all_choices"] = prediction_range
        res["correct_choice"] = sample["answer"]
        res["gt_content"] = options[ord(sample["answer"].upper()) - ord("A")]
    else:
        empty_prompt = SHORT_ANS_EXAMPLE_FORMAT.format(question)
        res["gt_content"] = parse_gold_answer(sample)

    res["final_input_prompt"] = (
        TASK_INSTRUCTIONS.strip() + "\n\n" + empty_prompt if TASK_INSTRUCTIONS else empty_prompt
    )
    return res


# ----------------------- 아래부터 eval_utils.py 원본 이식 -----------------------

def parse_multi_choice_response(response, all_choices, index2ans):
    for char in [",", ".", "!", "?", ";", ":", "'"]:
        response = response.strip(char)
    response = " " + response + " "

    index_ans = True
    ans_with_brack = False
    candidates = []
    for choice in all_choices:
        if f"({choice})" in response:
            candidates.append(choice)
            ans_with_brack = True

    if len(candidates) == 0:
        for choice in all_choices:
            if f" {choice} " in response:
                candidates.append(choice)

    if len(candidates) == 0 and len(response.split()) > 5:
        for index, ans in index2ans.items():
            if ans.lower() in response.lower():
                candidates.append(index)
                index_ans = False

    if len(candidates) == 0:
        pred_index = random.choice(all_choices)
    elif len(candidates) > 1:
        start_indexes = []
        if index_ans:
            if ans_with_brack:
                for can in candidates:
                    start_indexes.append(response.rfind(f"({can})"))
            else:
                for can in candidates:
                    start_indexes.append(response.rfind(f" {can} "))
        else:
            for can in candidates:
                start_indexes.append(response.lower().rfind(index2ans[can].lower()))
        pred_index = candidates[int(np.argmax(start_indexes))]
    else:
        pred_index = candidates[0]

    return pred_index


def check_is_number(string):
    try:
        float(string.replace(",", ""))
        return True
    except ValueError:
        return False


def normalize_str(string):
    string = string.strip()
    if check_is_number(string):
        string = string.replace(",", "")
        string = float(string)
        string = round(string, 2)
        return [string]
    else:
        string = string.lower()
        if len(string) == 1:
            return [" " + string, string + " "]
        return [string]


def extract_numbers(string):
    pattern_commas = r"-?\b\d{1,3}(?:,\d{3})+\b"
    pattern_scientific = r"-?\d+(?:\.\d+)?[eE][+-]?\d+"
    pattern_simple = r"-?(?:\d+\.\d+|\.\d+|\d+\b)(?![eE][+-]?\d+)(?![,\d])"
    return (re.findall(pattern_commas, string)
            + re.findall(pattern_scientific, string)
            + re.findall(pattern_simple, string))


def parse_open_response(response):
    def get_key_subresponses(response):
        key_responses = []
        response = response.strip().strip(".").lower()
        sub_responses = re.split(r"\.\s(?=[A-Z])|\n", response)
        indicators_of_keys = ["could be ", "so ", "is ",
                              "thus ", "therefore ", "final ", "answer ", "result "]
        for index, resp in enumerate(sub_responses):
            local_indicators = indicators_of_keys + ["="] if index == len(sub_responses) - 1 else indicators_of_keys
            shortest_key_response = None
            for indicator in local_indicators:
                if indicator in resp:
                    cand = resp.split(indicator)[-1].strip()
                    if not shortest_key_response or len(cand) < len(shortest_key_response):
                        shortest_key_response = cand
            if shortest_key_response and shortest_key_response.strip() not in [":", ",", ".", "!", "?", ";", ":", "'"]:
                key_responses.append(shortest_key_response)
        if len(key_responses) == 0:
            return [response]
        return key_responses

    key_responses = get_key_subresponses(response)
    pred_list = key_responses.copy()
    for resp in key_responses:
        pred_list.extend(extract_numbers(resp))

    tmp_pred_list = []
    for p in pred_list:
        tmp_pred_list.extend(normalize_str(p))
    return list(set(tmp_pred_list))


def eval_multi_choice(gold_i, pred_i):
    if isinstance(gold_i, list):
        return any(answer == pred_i for answer in gold_i)
    return gold_i == pred_i


def eval_open(gold_i, pred_i):
    if isinstance(gold_i, list):
        norm_answers = []
        for answer in gold_i:
            norm_answers.extend(normalize_str(str(answer)))
    else:
        norm_answers = normalize_str(str(gold_i))
    correct = False
    for pred in pred_i:
        if isinstance(pred, str):
            for norm_ans in norm_answers:
                if isinstance(norm_ans, str) and norm_ans in pred:
                    correct = True
                    break
        else:
            if pred in norm_answers:
                correct = True
        if correct:
            break
    return correct

print("MMMU 공식 파싱/채점 로직 로드 완료")


MMMU 공식 파싱/채점 로직 로드 완료


## 9. 이미지 처리 및 vLLM chat content 구성

1. **면적 상한 적용** — `IMAGE_MAX_PIXELS`를 넘는 이미지만 종횡비를 유지한 채 축소합니다
   (상한 이하 이미지는 원본 그대로). '고정 스펙 상수' 셀의 산정 근거 참조.
2. **`<image N>` 인터리빙** — 문항·보기 텍스트에 등장하는 `<image 1>`, `<image 2>` … 태그를
   등장 순서 그대로 실제 이미지로 치환해 텍스트/이미지가 교차된 content 리스트를 만듭니다
   (보기 안에 이미지가 들어간 문항도 커버).

In [ ]:
import base64
import io
import math
from PIL import Image

resize_counter = {"resized": 0, "total": 0}


def cap_image_pixels(img, max_pixels=IMAGE_MAX_PIXELS):
    """면적이 상한을 넘는 이미지만 종횡비 유지 축소."""
    resize_counter["total"] += 1
    if img.mode != "RGB":
        img = img.convert("RGB")
    w, h = img.size
    area = w * h
    if area <= max_pixels:
        return img
    scale = math.sqrt(max_pixels / area)
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    resize_counter["resized"] += 1
    return img.resize((new_w, new_h), Image.LANCZOS)


def _image_content(img):
    if IMAGE_CONTENT_MODE == "image_pil":
        return {"type": "image_pil", "image_pil": img}
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}


IMAGE_TAG_RE = re.compile(r"<image\s*(\d+)>")


def build_chat_content(sample, prompt_info):
    text = prompt_info["final_input_prompt"]
    content = []
    last_end = 0
    used_any_image = False

    for m in IMAGE_TAG_RE.finditer(text):
        if m.start() > last_end:
            seg = text[last_end:m.start()]
            if seg.strip():
                content.append({"type": "text", "text": seg})
        img = sample.get(f"image_{int(m.group(1))}")
        if img is not None:
            content.append(_image_content(cap_image_pixels(img)))
            used_any_image = True
        last_end = m.end()

    tail = text[last_end:]
    if tail.strip():
        content.append({"type": "text", "text": tail})

    # <image N> 태그가 없는데 이미지 필드가 존재하는 예외 케이스 방어
    if not used_any_image:
        for i in range(1, MAX_IMAGES_PER_QUESTION + 1):
            img = sample.get(f"image_{i}")
            if img is not None:
                content.append(_image_content(cap_image_pixels(img)))

    return content


def build_conversation(subject, row_idx):
    """배치 실행 시점에 샘플(이미지 포함)을 지연 로드하여 대화 1건을 구성."""
    sample = subject_datasets[subject][row_idx]
    prompt_info = construct_prompt(sample)
    content = build_chat_content(sample, prompt_info)
    conv = [{"role": "user", "content": content}]
    meta = {
        "id": sample["id"],
        "subject": subject,
        "question_type": sample["question_type"],
        "prompt_info": prompt_info,
    }
    return conv, meta

print("이미지/대화 구성 유틸 로드 완료")


이미지/대화 구성 유틸 로드 완료


## 10. 스모크 테스트 (본 실행 전 3문항 검증)

900문항을 돌리기 전에 **단일 이미지 객관식 / 다중 이미지 문항 / open 문항** 각 1건씩으로
모델 로드·프롬프트 구성·생성·파싱 경로를 먼저 검증합니다. 여기서 실패하면 본 실행은 하지 마세요.

In [ ]:
# 대표 문항 3건 선택 (컬럼 단위 조회라 이미지 디코딩 없음)
probe = {"single": None, "multi": None, "open": None}

for subject, row_idx, ex_id in flat_index:
    ds = subject_datasets[subject]
    if probe["single"] and probe["multi"] and probe["open"]:
        break
    q = ds["question"][row_idx]
    qt = ds["question_type"][row_idx]
    n_tags = len(set(IMAGE_TAG_RE.findall(q)))
    if probe["single"] is None and qt == "multiple-choice" and n_tags == 1:
        probe["single"] = (subject, row_idx, ex_id)
    if probe["multi"] is None and n_tags >= 2:
        probe["multi"] = (subject, row_idx, ex_id)
    if probe["open"] is None and qt != "multiple-choice":
        probe["open"] = (subject, row_idx, ex_id)

smoke_items = [v for v in probe.values() if v is not None]
print("스모크 대상:", [(s, eid) for s, _, eid in smoke_items])

smoke_convs, smoke_meta = [], []
for subject, row_idx, _ in smoke_items:
    c, m = build_conversation(subject, row_idx)
    smoke_convs.append(c)
    smoke_meta.append(m)

smoke_out = llm.chat(smoke_convs, sampling_params=sampling_params, use_tqdm=False)

for m, o in zip(smoke_meta, smoke_out):
    text = o.outputs[0].text
    n_prompt_tok = len(o.prompt_token_ids)
    print("=" * 90)
    print(f"[{m['id']}] subject={m['subject']} type={m['question_type']} "
          f"prompt_tokens={n_prompt_tok} gen_tokens={len(o.outputs[0].token_ids)}")
    print("-- 모델 출력 --")
    print(text[:500])
    if m["question_type"] == "multiple-choice":
        pred = parse_multi_choice_response(text, m["prompt_info"]["all_choices"], m["prompt_info"]["index2ans"])
        gold = m["prompt_info"]["correct_choice"]
        print(f"-- 파싱 결과: pred={pred} / gold={gold} / correct={eval_multi_choice(gold, pred)}")
    else:
        pred = parse_open_response(text)
        gold = m["prompt_info"]["gt_content"]
        print(f"-- 파싱 결과: pred={pred} / gold={gold} / correct={eval_open(gold, pred)}")

print("=" * 90)
print(f"이미지 리사이즈: {resize_counter['resized']}/{resize_counter['total']}장 "
      f"(면적 상한 {IMAGE_MAX_PIXELS:,} px 초과분만)")
print(f"프롬프트 토큰 최댓값(스모크): {max(len(o.prompt_token_ids) for o in smoke_out):,} "
      f"/ 프롬프트 예산 {MAX_MODEL_LEN - SAMPLING_RECIPE['max_tokens']:,}")


스모크 대상: [('Accounting', 'validation_Accounting_1'), ('Architecture_and_Engineering', 'validation_Architecture_and_Engineering_24'), ('Architecture_and_Engineering', 'validation_Architecture_and_Engineering_14')]
[validation_Accounting_1] subject=Accounting type=multiple-choice prompt_tokens=273 gen_tokens=398
-- 모델 출력 --
To determine the **per unit manufacturing overhead costs** when 30,000 units are produced, we must distinguish between **fixed** and **variable** manufacturing overhead.

From the table:

- **Fixed manufacturing overhead** = $6 per unit (this is *average* cost per unit at 25,000 units)
- **Variable manufacturing overhead** = $2 per unit

> Important: The question asks for **per unit manufacturing overhead costs** at 30,000 units. Since fixed costs do not change with volume (within relevant range
-- 파싱 결과: pred=B / gold=B / correct=True
[validation_Architecture_and_Engineering_24] subject=Architecture_and_Engineering type=multiple-choice prompt_tokens=480 gen_tokens=137

## 11. 본 추론 실행 (900문항)

- `BATCH_SIZE`(기본 50)단위로 실행하고 결과를 즉시 JSONL에 append 합니다.
- **문항 id 기반 재개**: 세션이 끊겨도 이 셀을 다시 실행하면 이미 완료된 문항은 건너뜁니다.
- 프롬프트가 예산을 초과하는 문항이 있으면 해당 문항 id를 경고로 출력합니다.

In [ ]:
import time

done_ids = set()
if os.path.exists(RAW_LOG_PATH):
    with open(RAW_LOG_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                done_ids.add(json.loads(line)["id"])
            except json.JSONDecodeError:
                continue
    print(f"기존 로그에서 완료된 문항 {len(done_ids)}건 확인 → 건너뜁니다.")

todo = [(s, r, eid) for (s, r, eid) in flat_index if eid not in done_ids]
print(f"이번 실행 대상: {len(todo)}문항")

n_batches = (len(todo) + BATCH_SIZE - 1) // BATCH_SIZE
t_start = time.time()

with open(RAW_LOG_PATH, "a", encoding="utf-8") as fout:
    for b in range(n_batches):
        lo, hi = b * BATCH_SIZE, min((b + 1) * BATCH_SIZE, len(todo))
        batch = todo[lo:hi]

        convs, metas = [], []
        for subject, row_idx, _ in batch:
            c, m = build_conversation(subject, row_idx)
            convs.append(c)
            metas.append(m)

        t0 = time.time()
        # use_tqdm=True: v3는 False라 배치당 최대 2시간 무출력이었고, 이것이 연결 끊김에 일조했습니다.
        outputs = llm.chat(convs, sampling_params=sampling_params, use_tqdm=True)
        dt = time.time() - t0

        for m, out in zip(metas, outputs):
            n_prompt_tok = len(out.prompt_token_ids)
            if n_prompt_tok > MAX_MODEL_LEN - SAMPLING_RECIPE["max_tokens"]:
                print(f"  [경고] {m['id']} 프롬프트 토큰 {n_prompt_tok:,}이 예산을 초과했습니다.")
            rec = {
                **m,
                "response": out.outputs[0].text,
                "n_prompt_tokens": n_prompt_tok,
                "n_gen_tokens": len(out.outputs[0].token_ids),
                "finish_reason": out.outputs[0].finish_reason,
            }
            fout.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
        fout.flush()

        elapsed = time.time() - t_start
        eta = elapsed / (b + 1) * (n_batches - b - 1)
        print(f"batch {b + 1}/{n_batches} [{lo}:{hi}] {dt:.1f}s | 누적 {elapsed/60:.1f}분 | ETA {eta/60:.1f}분")

print("\n전체 추론 완료 →", RAW_LOG_PATH)
print(f"이미지 리사이즈 누적: {resize_counter['resized']}/{resize_counter['total']}장")


이번 실행 대상: 900문항


Rendering conversations:   0%|          | 0/25 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 25/25 [09:56<00:00, 23.86s/it, est. speed input: 11.03 toks/s, output: 127.34 toks/s]


batch 1/36 [0:25] 596.7s | 누적 9.9분 | ETA 348.2분


Rendering conversations:   0%|          | 0/25 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 25/25 [05:16<00:00, 12.64s/it, est. speed input: 32.81 toks/s, output: 59.40 toks/s]


batch 35/36 [850:875] 316.4s | 누적 162.1분 | ETA 4.6분


Rendering conversations:   0%|          | 0/25 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 25/25 [00:05<00:00,  4.71it/s, est. speed input: 3781.26 toks/s, output: 26.57 toks/s]

batch 36/36 [875:900] 6.0s | 누적 162.2분 | ETA 0.0분

전체 추론 완료 → /content/drive/MyDrive/mmmu_qwen3vl4b_eval/v5_maxtok8192/raw_generations.jsonl
이미지 리사이즈 누적: 96/984장


## 12. 파싱 & 채점 (MMMU 공식 로직)

In [ ]:
import csv

graded = []
with open(RAW_LOG_PATH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        response = rec["response"]
        prompt_info = rec["prompt_info"]
        qtype = rec["question_type"]

        if qtype == "multiple-choice":
            pred = parse_multi_choice_response(response, prompt_info["all_choices"], prompt_info["index2ans"])
            gold = prompt_info["correct_choice"]
            correct = eval_multi_choice(gold, pred)
        else:
            pred = parse_open_response(response)
            gold = prompt_info["gt_content"]
            correct = eval_open(gold, pred)

        graded.append({
            "id": rec["id"],
            "subject": rec["subject"],
            "question_type": qtype,
            "pred": str(pred),
            "gold": str(gold),
            "correct": bool(correct),
            "n_prompt_tokens": rec.get("n_prompt_tokens"),
            "n_gen_tokens": rec.get("n_gen_tokens"),
            "finish_reason": rec.get("finish_reason"),
        })

assert len(graded) == 900, f"채점 대상이 900문항이 아닙니다: {len(graded)}. '본 추론 실행' 셀이 끝까지 완료됐는지 확인하세요."

csv_path = os.path.join(OUTPUT_DIR, "graded_results.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(graded[0].keys()))
    writer.writeheader()
    writer.writerows(graded)

print(f"채점 완료: {len(graded)}문항 → {csv_path}")
for row in graded[:5]:
    print(row)


채점 완료: 900문항 → /content/drive/MyDrive/mmmu_qwen3vl4b_eval/v5_maxtok8192/graded_results.csv
{'id': 'validation_Accounting_1', 'subject': 'Accounting', 'question_type': 'multiple-choice', 'pred': 'C', 'gold': 'B', 'correct': False, 'n_prompt_tokens': 273, 'n_gen_tokens': 225, 'finish_reason': 'stop'}
{'id': 'validation_Accounting_2', 'subject': 'Accounting', 'question_type': 'multiple-choice', 'pred': 'C', 'gold': 'C', 'correct': True, 'n_prompt_tokens': 209, 'n_gen_tokens': 8192, 'finish_reason': 'length'}
{'id': 'validation_Accounting_3', 'subject': 'Accounting', 'question_type': 'multiple-choice', 'pred': 'B', 'gold': 'B', 'correct': True, 'n_prompt_tokens': 147, 'n_gen_tokens': 750, 'finish_reason': 'stop'}
{'id': 'validation_Accounting_4', 'subject': 'Accounting', 'question_type': 'multiple-choice', 'pred': 'D', 'gold': 'D', 'correct': True, 'n_prompt_tokens': 345, 'n_gen_tokens': 307, 'finish_reason': 'stop'}
{'id': 'validation_Accounting_5', 'subject': 'Accounting', 'question_type

## 13. 과목별 / 전체 정확도 집계

In [ ]:
from collections import defaultdict
from datetime import datetime, timezone

by_subject = defaultdict(list)
for g in graded:
    by_subject[g["subject"]].append(g["correct"])

subject_acc = {s: sum(v) / len(v) for s, v in sorted(by_subject.items())}
overall_acc = sum(g["correct"] for g in graded) / len(graded)

print("=== 과목별 정확도 ===")
for subject, acc in subject_acc.items():
    print(f"{subject:40s} {acc * 100:6.2f}%  (n={len(by_subject[subject])})")

by_type = defaultdict(list)
for g in graded:
    by_type[g["question_type"]].append(g["correct"])
print("\n=== 문항 유형별 ===")
for qtype, v in by_type.items():
    print(f"{qtype:20s} {sum(v) / len(v) * 100:6.2f}%  (n={len(v)})")

n_len_capped = sum(1 for g in graded if g.get("finish_reason") == "length")
cap_ratio = n_len_capped / len(graded)
print(f"\n=== 생성 상한 도달 ===")
print(f"{SAMPLING_RECIPE['max_tokens']}토큰 상한 도달: {n_len_capped}/{len(graded)} ({cap_ratio * 100:.1f}%)")
if cap_ratio > 0.05:
    print("  [주의] 상한 도달 비율이 5%를 넘습니다. 해당 문항은 답을 끝맺지 못했을 수 있으므로")
    print("         max_tokens를 올려(RUN_TAG 변경 후) 재실행하는 것을 검토하세요.")
else:
    print("  상한 도달 비율이 낮아, 생성 예산 축소가 점수에 미친 영향은 미미합니다.")

print(f"\n=== 전체 정확도 (900문항) === {overall_acc * 100:.2f}%")

summary = {
    "model": MODEL_REPO,
    "model_revision": MODEL_REVISION,
    "dtype": "bfloat16",
    "dataset": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "split": DATASET_SPLIT,
    "n_total": len(graded),
    "overall_accuracy": overall_acc,
    "per_subject_accuracy": subject_acc,
    "per_question_type_accuracy": {k: sum(v) / len(v) for k, v in by_type.items()},
    "sampling_recipe": SAMPLING_RECIPE,
    "sampling_recipe_source": "https://github.com/QwenLM/Qwen3-VL (README.md, 'Evaluation Reproduction' section, Instruct models)",
    "sampling_recipe_deviation": {
        "field": "max_tokens",
        "official_value": OFFICIAL_MAX_TOKENS,
        "used_value": SAMPLING_RECIPE["max_tokens"],
        "reason": "원 지시사항에서 생성 예산은 자유 설정 항목. L4 24GB에서 900문항 완주를 위해 축소.",
        "n_finish_reason_length": n_len_capped,
        "ratio_finish_reason_length": cap_ratio,
    },
    "eval_protocol_source": "https://github.com/MMMU-Benchmark/MMMU (mmmu/utils/eval_utils.py, mmmu/utils/data_utils.py, mmmu/configs/llava1.5.yaml)",
    "inference_engine": "vLLM",
    "max_model_len": MAX_MODEL_LEN,
    "image_max_pixels": IMAGE_MAX_PIXELS,
    "images_resized": resize_counter["resized"],
    "images_total_seen": resize_counter["total"],
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
with open(os.path.join(OUTPUT_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary


=== 과목별 정확도 ===
Accounting                                63.33%  (n=30)
Agriculture                               56.67%  (n=30)
Architecture_and_Engineering              40.00%  (n=30)
Art                                       60.00%  (n=30)
Art_Theory                                80.00%  (n=30)
Basic_Medical_Science                     70.00%  (n=30)
Biology                                   56.67%  (n=30)
Chemistry                                 30.00%  (n=30)
Clinical_Medicine                         66.67%  (n=30)
Computer_Science                          56.67%  (n=30)
Design                                    73.33%  (n=30)
Diagnostics_and_Laboratory_Medicine       40.00%  (n=30)
Economics                                 66.67%  (n=30)
Electronics                               43.33%  (n=30)
Energy_and_Power                          60.00%  (n=30)
Finance                                   53.33%  (n=30)
Geography                                 53.33%  (n=30)
History        

{'model': 'Qwen/Qwen3-VL-4B-Instruct',
 'model_revision': 'ebb281ec70b05090aa6165b016eac8ec08e71b17',
 'dtype': 'bfloat16',
 'dataset': 'MMMU/MMMU',
 'dataset_revision': '98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68',
 'split': 'validation',
 'n_total': 900,
 'overall_accuracy': 0.5844444444444444,
 'per_subject_accuracy': {'Accounting': 0.6333333333333333,
  'Agriculture': 0.5666666666666667,
  'Architecture_and_Engineering': 0.4,
  'Art': 0.6,
  'Art_Theory': 0.8,
  'Basic_Medical_Science': 0.7,
  'Biology': 0.5666666666666667,
  'Chemistry': 0.3,
  'Clinical_Medicine': 0.6666666666666666,
  'Computer_Science': 0.5666666666666667,
  'Design': 0.7333333333333333,
  'Diagnostics_and_Laboratory_Medicine': 0.4,
  'Economics': 0.6666666666666666,
  'Electronics': 0.43333333333333335,
  'Energy_and_Power': 0.6,
  'Finance': 0.5333333333333333,
  'Geography': 0.5333333333333333,
  'History': 0.7,
  'Literature': 0.8,
  'Manage': 0.4666666666666667,
  'Marketing': 0.8333333333333334,
  'Material

## 14. 재현성 로그

In [ ]:
import sys
import vllm
import transformers
import datasets as datasets_lib
import PIL

env_info = {
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "vllm": vllm.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets_lib.__version__,
    "pillow": PIL.__version__,
    "gpu_name": gpu_name,
    "gpu_total_mem_gb": round(total_mem_gb, 1),
}
with open(os.path.join(OUTPUT_DIR, "environment.json"), "w", encoding="utf-8") as f:
    json.dump(env_info, f, ensure_ascii=False, indent=2)

env_info


{'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]',
 'torch': '2.13.0+cu130',
 'cuda': '13.0',
 'vllm': '0.29.0',
 'transformers': '5.16.1',
 'datasets': '4.8.5',
 'pillow': '12.2.0',
 'gpu_name': 'NVIDIA L4',
 'gpu_total_mem_gb': 22.0}

## 15. 결과 다운로드

`raw_generations.jsonl`(원본 응답) · `graded_results.csv`(문항별 채점) · `summary.json`(집계) · `environment.json`(환경 로그)

In [ ]:
zip_path = "/content/mmmu_qwen3vl4b_eval_results.zip"
!cd {OUTPUT_DIR} && zip -r -q {zip_path} .
print("zip:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Colab 환경이 아닙니다. 결과 위치:", OUTPUT_DIR)


zip: /content/mmmu_qwen3vl4b_eval_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>